In [1]:
"""
graficas_epidemicas.py
======================
Genera y exporta automáticamente las gráficas del modelo epidémico SIR
para COVID-19, Sarampión, SARS y Ébola.

Gráficas producidas (una carpeta por enfermedad + resúmenes comparativos):
  1. curvas_sir.png          — Compartimentos S, I, R a lo largo del tiempo
  2. ocupacion_hospitalaria.png — Camas ocupadas (general + UCI) vs. capacidad
  3. utilizacion_rho.png     — Utilización M/M/c ρ(t)
  4. comparacion_escenarios.png — 4 paneles: I, H, UCI, ρ por escenario
  5. sensibilidad_beta.png   — Barrido de β: pico I, pico H, tasa de ataque vs R₀
  (carpeta raíz)
  6. resumen_pico_H.png      — Barras: pico hospitalario por escenario y enfermedad
  7. resumen_dias_saturacion.png — Barras: días de saturación por escenario y enfermedad
  8. resumen_tasa_ataque.png — Barras: tasa de ataque por escenario y enfermedad

Uso:
    python graficas_epidemicas.py

Las imágenes se guardan en ./output_graficas/<enfermedad>/
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from scipy.integrate import solve_ivp
import pandas as pd
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# Configuración global
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR   = Path("output_graficas")
DPI          = 150
FIG_EXT      = "png"          # Cambiar a "pdf" o "svg" si se prefiere
HOSPITAL_BEDS = 8597
ICU_BEDS      = 453
N             = 18_125_000
T             = 200
t_span        = (0, T)
t_eval        = np.linspace(0, T, T * 2)

plt.rcParams.update({
    "figure.dpi":          DPI,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           True,
    "grid.alpha":          0.3,
    "font.size":           11,
    "savefig.bbox":        "tight",
    "savefig.dpi":         DPI,
})

# Colores por escenario (consistentes en todas las gráficas)
SCENARIO_COLORS = {
    "Base (sin intervención)":                  "#E24B4A",
    "Distanciamiento social (−45% contactos)":  "#378ADD",
    "Vacunación moderada (~60% cobertura)":     "#1D9E75",
    "Confinamiento estricto (−80% contactos)":  "#BA7517",
}
SCENARIO_LS = {
    "Base (sin intervención)":                  "-",
    "Distanciamiento social (−45% contactos)":  "--",
    "Vacunación moderada (~60% cobertura)":     "-.",
    "Confinamiento estricto (−80% contactos)":  ":",
}

# ─────────────────────────────────────────────────────────────────────────────
# Presets de enfermedades
# ─────────────────────────────────────────────────────────────────────────────

DISEASE_PRESETS = {
    "covid19": {
        "label":            "COVID-19 (cepa original)",
        "infectious_days":  10,
        "R0_empirical":     2.8,
        "p_hosp":           0.05,
        "p_icu":            0.01,
        "avg_stay_days":    10,
        "avg_icu_stay_days":14
    },
    "measles": {
        "label":            "Sarampión",
        "infectious_days":  8,
        "R0_empirical":     15.0,
        "p_hosp":           0.18,
        "p_icu":            0.05,
        "avg_stay_days":    5,
        "avg_icu_stay_days":10
    },
    "sars": {
        "label":            "SARS (2003)",
        "infectious_days":  7,
        "R0_empirical":     2.0,
        "p_hosp":           0.20,
        "p_icu":            0.07,
        "avg_stay_days":    14,
        "avg_icu_stay_days":21,
        "reference":        "Lipsitch et al. (2003); Riley et al. (2003)",
    },
    "ebola": {
        "label":            "Ébola (África Occidental 2014)",
        "infectious_days":  6,
        "R0_empirical":     1.8,
        "p_hosp":           0.80,
        "p_icu":            0.40,
        "avg_stay_days":    12,
        "avg_icu_stay_days":15,
        "reference":        "Berge et al. (2015); WHO Ebola Response Team (2014)",
    },
}

ESCENARIOS = [
    {"name": "Base (sin intervención)",                   "beta_mult": 1.00, "color": "#E24B4A", "ls": "-"},
    {"name": "Distanciamiento social (−45% contactos)",   "beta_mult": 0.55, "color": "#378ADD", "ls": "--"},
    {"name": "Vacunación moderada (~60% cobertura)",      "beta_mult": 0.40, "color": "#1D9E75", "ls": "-."},
    {"name": "Confinamiento estricto (−80% contactos)",   "beta_mult": 0.20, "color": "#BA7517", "ls": ":"},
]

# ─────────────────────────────────────────────────────────────────────────────
# Modelo SIR
# ─────────────────────────────────────────────────────────────────────────────

def sir_ode(t, y, beta, gamma, N):
    S, I, R = y
    fi = beta * S * I / N
    return [-fi, fi - gamma * I, gamma * I]


def run_sir(params, beta_override=None):
    beta  = beta_override if beta_override is not None else params["beta"]
    y0    = [params["S0"], params["I0"], 0]
    sol   = solve_ivp(
        sir_ode, t_span, y0,
        args=(beta, params["gamma"], params["N"]),
        t_eval=t_eval, method="RK45",
    )
    return pd.DataFrame({"t": sol.t, "S": sol.y[0], "I": sol.y[1], "R": sol.y[2]})


def compute_hospital_metrics(df, params, beta_override=None):
    df    = df.copy()
    beta  = beta_override if beta_override is not None else params["beta"]
    p_h, p_icu = params["p_hosp"], params["p_icu"]
    c, c_icu   = params["hospital_beds"], params["icu_beds"]
    mu, mu_icu = params["mu_hosp"], params["mu_icu"]

    df["H"]   = p_h   * df["I"]
    df["ICU"] = p_icu * df["I"]

    ni = beta * df["S"] * df["I"] / params["N"]
    df["lambda_hosp"] = p_h   * ni
    df["lambda_icu"]  = p_icu * ni
    df["rho_hosp"]    = df["lambda_hosp"] / (c * mu)
    df["rho_icu"]     = df["lambda_icu"]  / (c_icu * mu_icu)
    df["hosp_sat"]    = df["rho_hosp"] >= 1.0
    df["icu_sat"]     = df["rho_icu"]  >= 1.0
    return df


def build_params(preset):
    gamma = 1 / preset["infectious_days"]
    beta  = preset["R0_empirical"] * gamma
    return {
        "N":                N,
        "beta":             beta,
        "gamma":            gamma,
        "I0":               10,
        "S0":               N - 10,
        "p_hosp":           preset["p_hosp"],
        "p_icu":            preset["p_icu"],
        "hospital_beds":    HOSPITAL_BEDS,
        "icu_beds":         ICU_BEDS,
        "avg_stay_days":    preset["avg_stay_days"],
        "avg_icu_stay_days":preset["avg_icu_stay_days"],
        "mu_hosp":          1 / preset["avg_stay_days"],
        "mu_icu":           1 / preset["avg_icu_stay_days"],
    }


def run_all_scenarios(params):
    results = []
    for scen in ESCENARIOS:
        be  = params["beta"] * scen["beta_mult"]
        R0e = be / params["gamma"]
        df  = run_sir(params, beta_override=be)
        df  = compute_hospital_metrics(df, params, beta_override=be)
        results.append({"scenario": scen, "df": df, "beta": be, "R0": R0e})
    return results


def summarize(res, params):
    df  = res["df"]
    dt  = t_eval[1] - t_eval[0]
    idx = df["I"].idxmax()
    return {
        "Escenario":        res["scenario"]["name"],
        "R₀ efectivo":      round(res["R0"], 2),
        "Pico infectados":  int(df["I"].max()),
        "Día del pico":     round(df.loc[idx, "t"], 1),
        "Pico H":           int(df["H"].max()),
        "H supera cap.":    "Sí" if df["H"].max() > HOSPITAL_BEDS else "No",
        "Pico UCI":         int(df["ICU"].max()),
        "UCI supera cap.":  "Sí" if df["ICU"].max() > ICU_BEDS else "No",
        "ρ máx (hosp)":     round(df["rho_hosp"].max(), 3),
        "Días sat. hosp":   round(df["hosp_sat"].sum() * dt, 1),
        "Días sat. UCI":    round(df["icu_sat"].sum() * dt, 1),
        "Tasa ataque (%)":  round((N - df["S"].iloc[-1]) / N * 100, 1),
    }


def save(fig, path: Path, name: str):
    path.mkdir(parents=True, exist_ok=True)
    filepath = path / f"{name}.{FIG_EXT}"
    fig.savefig(filepath)
    plt.close(fig)
    print(f"  Guardado → {filepath}")

# ─────────────────────────────────────────────────────────────────────────────
# Gráfica 1: Curvas SIR base
# ─────────────────────────────────────────────────────────────────────────────

def plot_sir_curves(df_baseline, params, label, out_dir):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_baseline["t"], df_baseline["S"] / N,
            color="steelblue", lw=2, label="Susceptibles (S)")
    ax.plot(df_baseline["t"], df_baseline["I"] / N,
            color="tomato",    lw=2, label="Infectados (I)")
    ax.plot(df_baseline["t"], df_baseline["R"] / N,
            color="mediumseagreen", lw=2, label="Recuperados (R)")

    pico_t = df_baseline.loc[df_baseline["I"].idxmax(), "t"]
    pico_I = df_baseline["I"].max() / N
    ax.axvline(pico_t, color="tomato", ls="--", alpha=0.6,
               label=f"Día del pico: {pico_t:.0f}")
    ax.annotate(f"Pico\n{pico_I*100:.1f}%",
                xy=(pico_t, pico_I), xytext=(pico_t + 5, pico_I + 0.02),
                fontsize=9, color="tomato")

    ax.set_xlabel("Tiempo (días)")
    ax.set_ylabel("Fracción de la población")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title(f"Curvas SIR — {label}")
    ax.legend()
    fig.tight_layout()
    save(fig, out_dir, "curvas_sir")

# ─────────────────────────────────────────────────────────────────────────────
# Gráfica 2: Ocupación hospitalaria y UCI
# ─────────────────────────────────────────────────────────────────────────────

def plot_hospital_load(df_baseline, params, label, out_dir):
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    ax1 = axes[0]
    ax1.fill_between(df_baseline["t"], df_baseline["H"],
                     alpha=0.3, color="steelblue", label="Pacientes hospitalizados")
    ax1.plot(df_baseline["t"], df_baseline["H"], color="steelblue", lw=1.5)
    ax1.axhline(HOSPITAL_BEDS, color="tomato", ls="--", lw=2,
                label=f"Capacidad ({HOSPITAL_BEDS} camas)")
    ax1.fill_between(df_baseline["t"], df_baseline["H"], HOSPITAL_BEDS,
                     where=df_baseline["H"] > HOSPITAL_BEDS,
                     alpha=0.25, color="red", label="Zona de sobrecarga")
    ax1.set_ylabel("Camas ocupadas")
    ax1.set_title(f"Ocupación hospitalaria y de UCI — {label}")
    ax1.legend(loc="upper right")

    ax2 = axes[1]
    ax2.fill_between(df_baseline["t"], df_baseline["ICU"],
                     alpha=0.3, color="darkorange", label="Pacientes en UCI")
    ax2.plot(df_baseline["t"], df_baseline["ICU"], color="darkorange", lw=1.5)
    ax2.axhline(ICU_BEDS, color="tomato", ls="--", lw=2,
                label=f"Capacidad UCI ({ICU_BEDS} camas)")
    ax2.fill_between(df_baseline["t"], df_baseline["ICU"], ICU_BEDS,
                     where=df_baseline["ICU"] > ICU_BEDS,
                     alpha=0.25, color="red", label="Zona de sobrecarga UCI")
    ax2.set_xlabel("Tiempo (días)")
    ax2.set_ylabel("Camas de UCI ocupadas")
    ax2.legend(loc="upper right")

    fig.tight_layout()
    save(fig, out_dir, "ocupacion_hospitalaria")

# ─────────────────────────────────────────────────────────────────────────────
# Gráfica 3: Utilización ρ(t)
# ─────────────────────────────────────────────────────────────────────────────

def plot_utilization(df_baseline, label, out_dir):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_baseline["t"], df_baseline["rho_hosp"],
            lw=2, color="steelblue", label="Utilización hospitalaria ρ")
    ax.plot(df_baseline["t"], df_baseline["rho_icu"],
            lw=2, color="darkorange", label="Utilización UCI ρ")
    ax.axhline(1.0, color="tomato", ls="--", lw=2,
               label="Umbral de saturación (ρ = 1)")
    ax.fill_between(df_baseline["t"], df_baseline["rho_hosp"], 1.0,
                    where=df_baseline["rho_hosp"] > 1.0, alpha=0.15, color="steelblue")
    ax.fill_between(df_baseline["t"], df_baseline["rho_icu"], 1.0,
                    where=df_baseline["rho_icu"] > 1.0, alpha=0.15, color="darkorange")
    ax.set_xlabel("Tiempo (días)")
    ax.set_ylabel("Utilización ρ = λ / (c·μ)")
    ax.set_title(f"Utilización del sistema de salud ρ(t) — {label}")
    ax.legend()
    fig.tight_layout()
    save(fig, out_dir, "utilizacion_rho")

# ─────────────────────────────────────────────────────────────────────────────
# Gráfica 4: Comparación de escenarios (4 paneles)
# ─────────────────────────────────────────────────────────────────────────────

def plot_scenario_comparison(results, params, label, out_dir):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    panel_configs = [
        {"col": "I",        "ylabel": "Individuos infectados",  "title": "(a) Población infectada",    "hline": None},
        {"col": "H",        "ylabel": "Camas ocupadas",         "title": "(b) Ocupación hospitalaria", "hline": HOSPITAL_BEDS},
        {"col": "ICU",      "ylabel": "Camas UCI ocupadas",     "title": "(c) Ocupación de UCI",       "hline": ICU_BEDS},
        {"col": "rho_hosp", "ylabel": "Utilización ρ",          "title": "(d) Utilización ρ(t)",       "hline": 1.0},
    ]

    for ax, cfg in zip(axes, panel_configs):
        for res in results:
            scen = res["scenario"]
            df   = res["df"]
            ax.plot(df["t"], df[cfg["col"]],
                    color=scen["color"], ls=scen["ls"], lw=2,
                    label=f"{scen['name']} (R₀={res['R0']:.1f})")
        if cfg["hline"] is not None:
            ax.axhline(cfg["hline"], color="black", ls=":", lw=1.5,
                       label="Umbral de capacidad")
        ax.set_xlabel("Tiempo (días)")
        ax.set_ylabel(cfg["ylabel"])
        ax.set_title(cfg["title"])
        ax.legend(fontsize=8)

    fig.suptitle(f"Comparación de escenarios — {label}", fontsize=13, y=1.01)
    fig.tight_layout()
    save(fig, out_dir, "comparacion_escenarios")

# ─────────────────────────────────────────────────────────────────────────────
# Gráfica 5: Análisis de sensibilidad de β
# ─────────────────────────────────────────────────────────────────────────────

def plot_sensitivity(params, label, out_dir):
    beta_vals       = np.linspace(0.05, 0.50, 50)
    lista_pico_I    = []
    lista_pico_H    = []
    lista_ataque    = []

    for b in beta_vals:
        df = run_sir(params, beta_override=b)
        df = compute_hospital_metrics(df, params, beta_override=b)
        lista_pico_I.append(df["I"].max())
        lista_pico_H.append(df["H"].max())
        lista_ataque.append((N - df["S"].iloc[-1]) / N)

    R0_vals = beta_vals / params["gamma"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(R0_vals, lista_pico_I, color="tomato", lw=2)
    axes[0].axvline(1.0, color="gray", ls="--", label="R₀ = 1")
    axes[0].set_xlabel("Número básico de reproducción R₀")
    axes[0].set_ylabel("Pico de infectados")
    axes[0].set_title("Pico de infecciones vs R₀")
    axes[0].legend(fontsize=9)

    axes[1].plot(R0_vals, lista_pico_H, color="steelblue", lw=2)
    axes[1].axhline(HOSPITAL_BEDS, color="tomato", ls="--",
                    label=f"Capacidad = {HOSPITAL_BEDS}")
    axes[1].axvline(1.0, color="gray", ls="--")
    axes[1].set_xlabel("Número básico de reproducción R₀")
    axes[1].set_ylabel("Pico de hospitalizados")
    axes[1].set_title("Pico hospitalario vs R₀")
    axes[1].legend(fontsize=9)

    axes[2].plot(R0_vals, [a * 100 for a in lista_ataque],
                 color="mediumseagreen", lw=2)
    axes[2].axvline(1.0, color="gray", ls="--")
    axes[2].set_xlabel("Número básico de reproducción R₀")
    axes[2].set_ylabel("Tasa de ataque (%)")
    axes[2].set_title("Tasa de ataque total vs R₀")

    fig.suptitle(f"Análisis de sensibilidad: impacto de β — {label}",
                 fontsize=12, y=1.02)
    fig.tight_layout()
    save(fig, out_dir, "sensibilidad_beta")

# ─────────────────────────────────────────────────────────────────────────────
# Gráficas comparativas entre enfermedades (raíz de output)
# ─────────────────────────────────────────────────────────────────────────────

def plot_cross_disease_summary(all_results, all_labels):
    """Tres gráficas de barras agrupadas comparando las 4 enfermedades."""

    x       = np.arange(len(ESCENARIOS))
    width   = 0.20
    disease_colors = ["#E24B4A", "#378ADD", "#1D9E75", "#BA7517"]
    disease_keys   = list(all_results.keys())

    # ── Pico hospitalario ────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(13, 6))
    for i, key in enumerate(disease_keys):
        picos = [r["df"]["H"].max() for r in all_results[key]]
        bars  = ax.bar(x + i * width, picos, width, label=all_labels[key],
                       color=disease_colors[i], alpha=0.85)
    ax.axhline(HOSPITAL_BEDS, color="black", ls="--", lw=1.5,
               label=f"Capacidad hospitalaria ({HOSPITAL_BEDS} camas)")
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels([s["name"] for s in ESCENARIOS], rotation=12, ha="right", fontsize=9)
    ax.set_ylabel("Pico de camas ocupadas")
    ax.set_title("Pico de hospitalización por escenario e enfermedad")
    ax.legend(fontsize=9)
    fig.tight_layout()
    save(fig, OUTPUT_DIR, "resumen_pico_H")

    # ── Días de saturación ───────────────────────────────────────────────────
    dt = t_eval[1] - t_eval[0]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for i, key in enumerate(disease_keys):
        dias_hosp = [r["df"]["hosp_sat"].sum() * dt for r in all_results[key]]
        dias_icu  = [r["df"]["icu_sat"].sum()  * dt for r in all_results[key]]
        axes[0].bar(x + i * width, dias_hosp, width, label=all_labels[key],
                    color=disease_colors[i], alpha=0.85)
        axes[1].bar(x + i * width, dias_icu,  width, label=all_labels[key],
                    color=disease_colors[i], alpha=0.85)

    for ax, titulo in zip(axes, ["Días de saturación — Hospitalización", "Días de saturación — UCI"]):
        ax.set_xticks(x + width * 1.5)
        ax.set_xticklabels([s["name"] for s in ESCENARIOS], rotation=12, ha="right", fontsize=9)
        ax.set_ylabel("Días")
        ax.set_title(titulo)
        ax.legend(fontsize=8)
    fig.suptitle("Duración de saturación del sistema por escenario y enfermedad", fontsize=12)
    fig.tight_layout()
    save(fig, OUTPUT_DIR, "resumen_dias_saturacion")

    # ── Tasa de ataque ───────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(13, 6))
    for i, key in enumerate(disease_keys):
        tasas = [(N - r["df"]["S"].iloc[-1]) / N * 100 for r in all_results[key]]
        ax.bar(x + i * width, tasas, width, label=all_labels[key],
               color=disease_colors[i], alpha=0.85)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels([s["name"] for s in ESCENARIOS], rotation=12, ha="right", fontsize=9)
    ax.set_ylabel("Tasa de ataque (%)")
    ax.set_title("Tasa de ataque total por escenario y enfermedad")
    ax.legend(fontsize=9)
    fig.tight_layout()
    save(fig, OUTPUT_DIR, "resumen_tasa_ataque")

# ─────────────────────────────────────────────────────────────────────────────
# Ejecución principal
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print(f"\n{'='*60}")
    print("Generación de gráficas — Modelo epidémico SIR")
    print(f"{'='*60}\n")

    all_results = {}
    all_labels  = {}

    for disease_key, preset in DISEASE_PRESETS.items():
        label   = preset["label"]
        out_dir = OUTPUT_DIR / disease_key
        print(f"▶ {label}")

        params  = build_params(preset)
        results = run_all_scenarios(params)

        # Simulación base (primer escenario = multiplicador 1.0)
        df_baseline = results[0]["df"]

        # Gráficas individuales por enfermedad
        plot_sir_curves(df_baseline, params, label, out_dir)
        plot_hospital_load(df_baseline, params, label, out_dir)
        plot_utilization(df_baseline, label, out_dir)
        plot_scenario_comparison(results, params, label, out_dir)
        plot_sensitivity(params, label, out_dir)

        # Tabla de resumen en consola
        summary_rows = [summarize(r, params) for r in results]
        summary_df   = pd.DataFrame(summary_rows).set_index("Escenario")
        print(summary_df.to_string())
        print()

        all_results[disease_key] = results
        all_labels[disease_key]  = label

    # Gráficas comparativas entre enfermedades
    print("▶ Generando gráficas de resumen comparativo entre enfermedades...")
    plot_cross_disease_summary(all_results, all_labels)

    print(f"\n{'='*60}")
    print(f"Listo. Todas las imágenes guardadas en: {OUTPUT_DIR.resolve()}/")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    main()


Generación de gráficas — Modelo epidémico SIR

▶ COVID-19 (cepa original)
  Guardado → output_graficas/covid19/curvas_sir.png
  Guardado → output_graficas/covid19/ocupacion_hospitalaria.png
  Guardado → output_graficas/covid19/utilizacion_rho.png
  Guardado → output_graficas/covid19/comparacion_escenarios.png
  Guardado → output_graficas/covid19/sensibilidad_beta.png
                                         R₀ efectivo  Pico infectados  Día del pico  Pico H H supera cap.  Pico UCI UCI supera cap.  ρ máx (hosp)  Días sat. hosp  Días sat. UCI  Tasa ataque (%)
Escenario                                                                                                                                                                                        
Base (sin intervención)                         2.80          4985935          83.7  249296            Sí     49859              Sí        38.617            69.7           99.2             92.4
Distanciamiento social (−45% contactos)        